# CS1 Homework 7

必ず，以下の変数を自分の学籍番号に変更して下さい．

In [ ]:
my_student_ID = "26B20988"

## 内容


この自習用ノートブックは，いくつかのソートアルゴリズムの性能を比較するプロジェクトです．

* 実行時間の計測
* ランダムなワークロード（入力データ）の作成
* データのCSVファイルへの集計
* 折れ線グラフの作成
* データの分析

## 実行時間の計測


Pythonの`time`モジュールは，時間を計測するための様々な関数を提供しています． プログラムの実行時間の計測には，高い精度で時間を計測できる関数`perf_counter()`が使えます．この関数は，ある起点からの経過時間を秒単位で計測し，少数（浮動小数点数）で返します．

In [ ]:
import time

assert my_student_ID != "99B99999", "Please set your student ID in the variable my_student_ID at the beginning of this notebook."

# record starting time
start_time = time.perf_counter() # time in seconds (start)
# ↓↓↓ BEGIN CODE TO MEASURE
for i in range(1000):
    for j in range(1000):
        pass
# ↑↑↑ END CODE TO MEASURE
# record ending time
end_time = time.perf_counter() # time in seconds (end)
# display the time difference in milliseconds
print(f"runtime: {1000 * (end_time - start_time)} ms")

命令 `pass`は，何もしない命令です．この例では，何もしない２重ループの実行時間を計測しています．何か別のコードの実行時間を計測してみてください．

## ランダムなワークロードの作成


実験のために，入力として与える様々な長さのランダムなリストを作成する必要があります．

すべてのランダムなリストを事前に作成し，同じリストのデータセットをそれぞれのソートアルゴリズムに適用します．
これは，同じ条件でそれぞれのソートアルゴリズムの性能を計測し，実験時間の変動を減らすためです．

Pythonでは，モジュール`random`がランダムな数やリストのランダムな並び替えを生成する関数を提供しています．
下記のコードは，0から9のランダムなリストを重複ありと重複なしで生成する方法を示しています．

In [ ]:
import random

assert my_student_ID != "99B99999", "Please set your student ID in the variable my_student_ID at the beginning of this notebook."

my_list = list(range(10))
print("original list:", my_list)

# sample (no repetitions)
sampled_list = random.sample(my_list, 10)
print("sampled list :", sampled_list)

# shuffle (no repetitions)
shuffled_list = my_list[:]
random.shuffle(shuffled_list)
print("shuffled list:", shuffled_list)

# random tries (repetitions)
random_list = [ random.randint(0,9) for _ in range(10) ]
print("random list  :", random_list)

オプション課題として，上記のどの方法が高速か比べてみてください．下に新たにコードブロックを作成し，コードをコピーし，実験のためにコードを書き換えてみてください．

## データの記録


### 補助関数


ソートアルゴリズムや性能の計測で用いる補助関数をいくつか準備します．６回目に定義した関数`merge`を思い出してください．関数`create_random_list`は，shuffle methodを用いて重複のないランダムリストを生成します． 関数`timed_sort`は，与えられたソートの関数をデータに適用するとともに，実行時間の計測を行います．

In [ ]:
import random
import time

assert my_student_ID != "99B99999", "Please set your student ID in the variable my_student_ID at the beginning of this notebook."


def create_random_list(n):
    """Creates a list of length n as a random permutation of the values from 0 to n-1."""
    if n < 0: raise ValueError("negative size")
    shuffled_list = list(range(n))
    random.shuffle(shuffled_list)
    return shuffled_list


def timed_sort(sort_fct, data):
    """Given a sort function and a list, measures the time (in ms) it takes to sort the data.
    An error occurs if the "sorted" list is not actually sorted.
    """
    #
    # copy the data to protect it from modification of its content
    #
    workload = data[:]
    #
    # measure runtime
    #
    start = time.perf_counter() # time in seconds (start)
    sorted_data = sort_fct(workload)
    end = time.perf_counter()   # time in seconds (end)
    #
    # check result
    #
    if sorted_data != sorted(data):
        raise ValueError("Error in sorting. Result after sorting attempt: ", sorted_data)
    #
    # return runtime [ms]
    #
    return (end - start) * 1000 # as milliseconds


def merge(a, b):
    """Returns a sorted list with the elements of two sorted lists a and b."""
    merged = []
    i = 0
    j = 0
    while i<len(a) and j<len(b):
        if a[i] < b[j]:
            merged.append(a[i])
            i += 1
        else:
            merged.append(b[j])
            j += 1
    return merged + a[i:] + b[j:]

`timed_sort`以外の関数については説明はいらないと思います．関数`timed_sort`は，二つの引数をとります: `sort_fct`はソートを行う関数で，`data`はソートするリストです．

関数を引数として渡すことができることに注意してください． 例えば，`timed_sort(mergesort,...)`を実行すると, `timed_sort`が`sort_fct`を呼び出した時に，関数 `mergesort`が実行されます． `timed_sort`を使うことで，任意のソート関数`sort_fct`と任意の`data`について，実行時間が計測できるようになり，とても便利です．

関数`timed_sort` は，関数`sort_fct`のリスト `data`に対する実行時間を計測します．その後に，結果のリストが実際に整列済みであるかを確認し，そうでなければエラーになります． 最後に，実行時間をミリ秒で測った値を返します． ワークロードが変更されれないように，最初に，`data`のコピーを作っていることに注意してください．

### ソート関数


下のプログラムは，これから分析するソート関数を定義しています．公平な比較のため，どのソート関数もコピーしたリストで動作し，引数として受け取るリスト`data` は変更しません． 関数`timed_sort`でもコピーをしているので，余計に見えるかもしれませんが，二つのコードは独立しているので，それぞれコピーをすることで，より頑強で個別に検証しやすくなります． `timed_sort`で行うコピーはワークロードのデータが変更されるのを防ぐためで，ソート関数で行うコピーは公平な比較のためです．

In [ ]:
assert my_student_ID != "99B99999", "Please set your student ID in the variable my_student_ID at the beginning of this notebook."


def insertion_sort(data):
    """
    Returns a sorted copy of an list. (insertion sort)
    """
    output = data[:]
    for i in range(len(output)):
        for j in range(i-1, -1, -1):
            if output[j] > output[j+1]:
                # swap
                output[j], output[j+1] = output[j+1], output[j]
            else:
                break # <- for j の繰り返しをいきなり止める
    return output


def quicksort(data):
    """
    Returns a sorted copy of a list. (quicksort)
    """
    if len(data)  < 2: return data[:]
    if len(data) == 2: return [min(data), max(data)]
    #
    pivot = data[0]
    tail  = data[1:]
    left  = [ v for v in tail if v < pivot ]
    right = [ v for v in tail if v >= pivot ]
    return quicksort(left) + [ pivot ] + quicksort(right)


def mergesort(data):
    """
    Returns a sorted copy of an list. (merge sort)
    """
    if len(data) < 2: return data[:]
    low = 0
    mid = len(data) // 2
    hi  = len(data)
    left  = mergesort(data[low:mid])
    right = mergesort(data[mid:hi])
    return merge(left, right)


#### Bonuses ####

from heapq import heappush, heappop, heapify

def heapsort(data):
    lst = data[:]
    # TODO: implement the heapsort algorithm
    output = [0] * len(data)
    heapify(lst)
    for i in range(len(data)):
        output[i] = heappop(lst)
    return output


def bubblesort(data):
    output = data[:]
    # TODO: implement the bubblesort algorithm
    n = len(output)
    for k in range(n-1, 0, -1): #  Execute the body for k = n-1, n-2, ..., 1
        for i in range(k):
            if output[i]>output[i+1]:
                output[i], output[i+1] = output[i+1], output[i]
    return output


def selection_sort(data):
    output = data[:]
    # TODO: implement the selection sort algorithm
    for i in range(len(data)):
      min = i
      for j in range(i+1, len(data)):
        if output[min]>output[j]:
          min = j
      output[i],output[min] = output[min],output[i]
    return output

自分で書いたソート関数を上のブロックに追加しても良いです．

### 実行時間の計測


下のコードは，これまでに定義した関数を使って実験を行います．実験の主なパラメータが最初に定義されています．

* `ALGORITHMS` は実験するアルゴリズのリストで，リストの各要素は三つ組です．_algorithm_　はアルゴリズムの名前，_color_ はデータをプロットするときの色， _sort function_ は実行するソート関数です．
* `REPETITIONS`には，与えられたリストの各大きさについて，何個の異なるリストを生成するかを指定します．
* `TEST_SIZES` は，ワークロードとして生成するリストの大きさのリストです．

結果として，`REPETITIONS * len(TEST_SIZES)`個の異なるランダムリストがワークロードとして生成されます．それぞれのリストに対して，指定されたソートアルゴリズムが実行されます．よって， 関数 `timed_sort`は， `REPETITIONS * len(TEST_SIZES) * len(ALGORITHMS)`回呼ばれることになります．

実行時間の情報は，下記の情報からなるリストとして記録されます．

* `'algorithm'` は，アリゴリズムの名前
* `'size'` は入力リストの長さ
* `'repetition'` は繰り返しの回数(between 0 and `REPETITIONS-1`)
* `'runtime(ms)'` は，計測された実行時間（ミリ秒）
* `'color'` は，アルゴリズムをプロットする時に使われる色

データは，上記を見出しとするCSV fileに保存されます．プログラムの実行後に，CSVファイルを確認してください．

In [ ]:
import csv
import sys

assert my_student_ID != "99B99999", "Please set your student ID in the variable my_student_ID at the beginning of this notebook."

DATA_FILENAME = 'runtime_data.csv'

#
# parameters of the experiment
# (NB: you can customize them)
#

ALGORITHMS = [
    # Algorithm name,   Color,          Algorithm function
    ("Selection sort",  "darkorchid",   selection_sort ),
    ("Insertion sort",  "lightseagreen",   insertion_sort ),
    ("Bubble sort",     "royalblue",    bubblesort ),
    ("Heapsort",        "lightcoral",   heapsort),
    ("Quicksort",       "orange",       quicksort ),
    ("Merge sort",      "red",          mergesort ),
    ("Builtin sort",    "black",        sorted )
]

REPETITIONS  = 8                    # TODO: increase for more stable statistics

TEST_SIZES  = [ 10, 20, 40, 80, 160, 320, 640, 1280,
#               2560,               # TODO: uncomment this line to enable bigger sizes
]

#
# create the workload (random lists for each of the test sizes)
#
workload = [
    ( size, [ create_random_list(size) for _ in range(REPETITIONS) ] )
        for size in TEST_SIZES
]

#
# extend recursion limit to avoid potential errors when sorting large lists with quicksort
sys.setrecursionlimit(max(1000, max(TEST_SIZES)*3//2))

#
# sort and gather execution data
#
runtime_data_headers = ['algorithm', 'size', 'repetition', 'runtime(ms)', 'color']
runtime_data = []
for k, (name, col, sort) in enumerate(ALGORITHMS):
    print("- ", name)

    for i, (size, lists) in enumerate(workload):
        print(f"  {size},", end="", flush=True)
        #
        # 2. run the algorithms
        for i, test_list in enumerate(lists):
            runtime = timed_sort(sort, test_list)
            runtime_data.append({
                'algorithm': name,
                'size': size,
                'repetition': i,
                'runtime(ms)': runtime,
                'color':col
            })
    print()

#
# write the data to a CSV file
#
with open(DATA_FILENAME, 'w') as f:
    writer = csv.DictWriter(f, fieldnames=runtime_data_headers)
    writer.writeheader()
    writer.writerows(runtime_data)

print("Completed.")

## グラフの生成


実行時間のデータを集計したので，ここからはデータの可視化を行います．(これ以降を続ける前に，これまでのブロックを実行する必要があります．) 下のプログラムはpython のライブラリ[matplotlib](https://matplotlib.org/) とその拡張[seaborn](https://seaborn.pydata.org/) をデータのプロットのために，データの操作のために[pandas](https://pandas.pydata.org/docs/)を使います．

> (注意) 必要なコードは書いてあるので，matplotlib や seaborn を学ぶ必要はありません．将来，データ処理を研究や仕事で行う必要がある場合は，これらのライブラリに少しずつなれておくと役立つと思います．

下のプログラムは4つのステップで構成されています．
1. 実行時間に関するデータをCSVファイルから読み込みます．
2. データを用いてアルゴリズム名から色への対応のテーブルを作成します．
3. データの大きさと実行時間の対応を示す折れ線グラフを作成します．
4. グラフをディスプレーに表示し，PDFファイルに保存します．

各アルゴリズムとデータの大きさについて，ドットは同じ長さのリストに対する実行時間の中央値で(`estimator="median"`)，その周辺の色が付いた領域が四分位範囲 (`errorbar=("pi",50)`)です．

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA_FILENAME = 'runtime_data.csv'

#
# load runtime data
#
df = pd.read_csv(DATA_FILENAME)

#
# compute color palette
#
algorithm_colors = df[['algorithm', 'color']].drop_duplicates()
palette = {
    algorithm: color
    for algorithm, color in algorithm_colors.values
}

#
# (lin-lin) plot the medians and the inter-quartile ranges
#
ax = sns.lineplot(
    x='size',
    y='runtime(ms)',
    hue='algorithm', style='algorithm',
    palette=palette,
    markers=True, dashes=False,
    data=df,
    estimator="median",
    errorbar=("pi", 50), # 1st and 3rd quartiles
)
ax.set(title=f'Runtime of Sort Algorithms; linear ({my_student_ID})')
ax.margins(0)
ax.grid()

#
# show the chart
#
plt.tight_layout()
plt.show()

どちらの軸に関しても線形のスケールです．実行時間が数桁異なることもあり，線形スケールでは比較が困難です．データのが大きくなるにつれて差が非常に大きくなるのは分かりますが，全体像を把握するのは困難です．

次に対数スケールのグラフを作ります．対数スケールでは具体的な実行時間を把握しづらいですが，増加の度合いをデータの大きさを気にせずに比較できます．

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import math

assert my_student_ID != "99B99999", "Please set your student ID in the variable my_student_ID at the beginning of this notebook."

DATA_FILENAME = 'runtime_data.csv'

#
# load runtime data
#
df = pd.read_csv(DATA_FILENAME)

#
# compute color palette
#
algorithm_colors = df[['algorithm', 'color']].drop_duplicates()
palette = {
    algorithm: color
    for algorithm, color in algorithm_colors.values
}

#
# reset settings to original
#
sns.reset_orig()

#
# (log-log) plot the medians and the inter-quartile ranges
#
ax = sns.lineplot(
    x='size',
    y='runtime(ms)',
    hue='algorithm', style='algorithm',
    palette=palette,
    markers=True, dashes=False,
    data=df,
    estimator="median",
    errorbar=("pi", 50), # 1st and 3rd quartiles
)
# configure plot
ax.set(xscale='log', yscale='log',
       title=f'Runtime of Sort Algorithms; log scale ({my_student_ID})'
)
ax.margins(0)
ax.grid()

#
# draw the reference slopes
#
def n_squared(n): return n ** 2
def n_log_n(n): return n * math.log(n)
def log_n(n): return math.log(n)
END_INDEX = 2
END_SIZE = TEST_SIZES[END_INDEX]
SLOPE_SIZES = [ TEST_SIZES[0], END_SIZE ]
# * O(n log n)
ax.plot(SLOPE_SIZES, [ 1, n_log_n(END_SIZE) / n_log_n(TEST_SIZES[0]) ], dashes=(1,2), alpha=0.25, color='0.2')
ax.text(END_SIZE, n_log_n(END_SIZE) / n_log_n(TEST_SIZES[0]), "$O(n \\log n)$", alpha=0.5, color='0.2', horizontalalignment='left')
# * O(n^2)
ax.plot(SLOPE_SIZES, [ 1, n_squared(END_SIZE) / n_squared(TEST_SIZES[0]) ], alpha=0.25, dashes=(1,2), color='0.2')
ax.text(END_SIZE, n_squared(END_SIZE) / n_squared(TEST_SIZES[0]), "$O(n^2)$", alpha=0.5, color='0.2', horizontalalignment='left')
# * O(n)
ax.plot(SLOPE_SIZES, [ 1, END_SIZE / TEST_SIZES[0] ], alpha=0.25, dashes=(1,2), color='0.2')
ax.text(END_SIZE, END_SIZE / TEST_SIZES[0], "$O(n)$", alpha=0.5, color='0.2', horizontalalignment='left')
# * O(log n)
ax.plot(SLOPE_SIZES, [ 1, log_n(END_SIZE) / log_n(TEST_SIZES[0]) ], alpha=0.25, dashes=(1,2), color='0.2')
ax.text(END_SIZE, log_n(END_SIZE) / log_n(TEST_SIZES[0]), "$O(\\log n)$", alpha=0.5, color='0.2', horizontalalignment='left')

#
# save and show the chart
#
plt.tight_layout()
plt.savefig("sort_comparison_log_log.pdf")
plt.show()

データを再確認します．縦軸の１目盛は実行時間の10倍の増加を表しています．同様に，横軸の１目盛はデータの大きさが10倍になることを表しています．折れ線の傾きは入力データの大きさに対する実行時間の増え方を表しています．

参考として, $O(n)$, $O(n\log n)$, $O(n^2)$に対応するグラフを点線で示しています．各線の縦方向の位置は無意味で，傾きのみが重要です．

## 解析（分析)


最後にグラフを分析します．

グラフから，insertion sortとselection sortの傾きは$O(n^2)$にほぼ平行であり，一方でquicksort，mergesort，builtin sortは$O(n \log n)$に追従していることが見て取れます．また，builtin sortは他のどのアルゴリズムよりも少なくとも10倍は速いことも分かります．さらに，$O(n \log n)$の傾きは，$O(n)$の傾きと比べてそれほど急ではないということも観察できます．入力データのサイズが小さいときは，（builtinを除いて）アルゴリズム間の差がわずか（マージナル）であることも確認できます．

他にも確実に多くの追加の考察ができるはずです．ここから，性能評価プロジェクトの最も興味深く有益な部分が始まります．あなたの観察結果を踏まえて，以下の点について考えてみてください：
* これらの観察から，どのような知見（インサイト）を得ることができますか？
* どのような影響や帰結（インプリケーション）が考えられますか？
* この分析の限界や欠点は何でしょうか？
* これにより，さらにどのような疑問が湧いてきますか？
* 結論として導き出せる主なポイントは何でしょうか？


## 提出方法


次の手順で提出してくさい．提出方法に注意してください.

1. ノートブックを自分のドライブにコピーしてください．
1. __注意：__ `my_student_ID`の変数に，自分の学籍番号を書き直してください．
1. 説明を読んでください．
1. 実験を実行してください．
1. グラフを分析してください．
1. LMS から得られたグラフを提出してください：`sort_comparison_log_log.pdf` (本ノートブック)．
1. チャートを見て得られる主な観察結果（ソートアルゴリズムの比較として）をリストアップした短いレポートを **1ページ限り** のPDFファイルとして作成してください．レポートは **簡潔**な 箇条書き（例えば、LaTeXの`itemize`環境を使用）で記述します．ファイル名は「`observations.pdf`」としてください．レポートには必ず，タイトル，作者名，学籍番号を含む．
1. 本ノートブックを完成したら，「出力をすべて消去」をして，`ipynb` ファイルとしてダウンロードして，ファイルを提出して下さい．
1. LMS から三つの要求ファイルを提出してください．

### Bonus

工夫なしは最高８０点となります．工夫点には，以下のアルゴリズムを追加して，解析します．

1. 課題`hw6`からの`bubblesort`を追加する． _(+5点)_
1. 課題`hw6`からの`heapsort`を追加する． _(+5点)_
1. 選択ソート(selection sort)をしらべて，`selection_sort`を書いて，追加する． _(+10点)_

注意：

* アルゴリズムを追加するには，変数リスト`ALGORITHMS`の定義に追加したいアリゴリズムに関するコメントを外して下さい
* すべてのセルを再度実行してください．colabでは，メニューから  _"Runtime" > "Run all"_ を得れべば良いです．しばらく待ってください...

# End